In [170]:
import os
from dotenv import load_dotenv
load_dotenv()


groq_api_key = os.environ.get("GROQ_API_KEY") 


from langchain_groq import ChatGroq
model=ChatGroq(model= "llama-3.1-8b-instant", groq_api_key=groq_api_key)

In [171]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hey, I am Mohammad and i am Data Scientist.")])

AIMessage(content="Hello Mohammad! It's nice to meet you. As a Data Scientist, I'm sure you're always eager to dive into the world of data, uncover hidden patterns, and make informed decisions. What kind of projects or areas of expertise are you currently working on or interested in?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 46, 'total_tokens': 103, 'completion_time': 0.089007722, 'completion_tokens_details': None, 'prompt_time': 0.004363549, 'prompt_tokens_details': None, 'queue_time': 0.077816302, 'total_time': 0.093371271}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_dc4e569a6a', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c969b-98ab-7911-8cd0-5cad0c35bac9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 46, 'output_tokens': 57, 'total_tokens': 103})

In [172]:
from langchain_core.messages import AIMessage

model.invoke(
    [HumanMessage(content="Hey, I am Mohammad and i am Machine Learning Engineer.")
     , AIMessage(content="Nice to meet you, Mohammad. As a Data Scientist, you must be working with various data sets, developing predictive models, and extracting valuable insights from the data. What areas of data science interest you the most, such as machine learning, natural language processing, or deep learning?")
     ,HumanMessage(content="Hey what is my name and what do i do?")
      
     
     ])


AIMessage(content='Your name is Mohammad, and you are a Machine Learning Engineer.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 124, 'total_tokens': 138, 'completion_time': 0.009769569, 'completion_tokens_details': None, 'prompt_time': 0.007839791, 'prompt_tokens_details': None, 'queue_time': 0.061716068, 'total_time': 0.01760936}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_8f8420ecd7', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c969b-9a35-78a2-a007-14cfc5f31f50-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 124, 'output_tokens': 14, 'total_tokens': 138})

### Message History

We can use a Message History wrapper to make our model stateful.

This component automatically:

- Stores user inputs  
- Stores model responses  
- Persists conversation history in a datastore  

When a new interaction occurs:

- Previous messages are retrieved  
- They are injected back into the chain  
- The model receives full conversational context  

This enables multi-turn conversations where the model remembers prior exchanges instead of treating each request independently.

In short:

Message History → Memory Layer  
It allows the chain to maintain conversational continuity.

In [173]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


store = {}
def get_session_history(session_id:str)-> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [174]:
config = {"configurable": {"session_id": "abc123"}}

In [175]:
response = with_message_history.invoke(
    [HumanMessage(content="Hey, I am Mohammad and i am Machine Learning Engineer")],
    config=config
)

In [176]:
print(response.content)

Nice to meet you, Mohammad. As a Machine Learning Engineer, you're likely working on building and deploying intelligent systems that can learn and improve from data. What specific areas of Machine Learning are you interested in or working on? Are you into deep learning, natural language processing, computer vision, or something else?


In [177]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config
)
response.content

"Your name is Mohammad, and you're a Machine Learning Engineer."

### changing the session id to see the difference in response

In [178]:
config1 = {"configurable": {"session_id": "xyz789"}}
response1 = with_message_history.invoke(
    [HumanMessage(content="What's my name?")], config=config1)

response1.content

"I don't have any information about your name. I'm a large language model, I don't have the ability to store or recall personal information about individual users. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you!"

## Prompt templates with history

In [179]:

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


prompt = ChatPromptTemplate.from_messages(

    [
        ("system", "You are a helpful AI assistant. Answer all the questions to the best of your ability"),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [180]:
chain.invoke({"messages": [HumanMessage(content="Hey, I am Mohammad and i am Machine Learning Engineer")]})

AIMessage(content="Nice to meet you, Mohammad. As a Machine Learning Engineer, you must be excited about the latest advancements in AI and ML. I'm here to help and provide you with any information, resources, or assistance you might need.\n\nWhat brings you here today? Are you working on a specific project, or do you have a question about a particular aspect of Machine Learning or AI?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 63, 'total_tokens': 140, 'completion_time': 0.112204928, 'completion_tokens_details': None, 'prompt_time': 0.004576898, 'prompt_tokens_details': None, 'queue_time': 0.061251003, 'total_time': 0.116781826}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d317489708', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c969b-b0ea-7b83-aeb9-f5806232282e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 63, 'o

In [181]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [182]:
config = {"configurable": {"session_id": "chat3"}}
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="Hey, I am Mohammad and i am Machine Learning Engineer")]}, config=config)
response

AIMessage(content="Nice to meet you, Mohammad. As a Machine Learning Engineer, you're probably always on the lookout for new techniques and tools to improve your skills. What's been the most interesting or challenging project you've worked on recently?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 63, 'total_tokens': 109, 'completion_time': 0.099828622, 'completion_tokens_details': None, 'prompt_time': 0.029011222, 'prompt_tokens_details': None, 'queue_time': 0.074059967, 'total_time': 0.128839844}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_8f8420ecd7', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c969b-b35e-7b13-9217-7cc04e3c2b8f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 63, 'output_tokens': 46, 'total_tokens': 109})

In [183]:
## Adding more complexity to the prompt


prompt = ChatPromptTemplate.from_messages(
    [

        (
            "system", "You are an helpful AI assistant. Answer all the questions to the best of your ability in {language} language"
         ),
        MessagesPlaceholder(variable_name="messages"),
    ] )

chain = prompt | model

In [184]:
response = chain.invoke({"messages":[HumanMessage(content="Hello, my name is Mohammad")], "language":"Darija"})
response.content

'مرحباً بني Mohammad, خوش به حالت. چيستاني داريد؟'

In [185]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [186]:
config7 = {"configurable": {"session_id": "chat4"}}
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="What is my name?")], "language": "French"}, config=config7
)
response.content

'Je suis désolé, mais je ne connais pas votre nom. Puis-je vous aider à trouver votre nom ou à vous poser une question qui pourrait vous aider à vous rappeler?'

In [187]:
response = with_message_history.invoke(
    {
        "language": "French",
        "messages": [HumanMessage(content="What is my name?")]}, config=config7)
response.content

"Désolé, mais je ne connais pas votre nom. Puis-je vous parler de quelque chose d'autre ou vous aider à trouver une information en particulier ?"

## Managing the conversation history
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [206]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(max_tokens=400, strategy = "last", token_counter=model,
                        include_system=True, allow_partial=False,
                        start_on = "human")

In [207]:
messages1 = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages1)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="hi! I'm bob", additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs

In [208]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough


prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system", "You are an helpful AI assistant. Answer all the questions to the best of your ability in {language} language"
         ),
        MessagesPlaceholder(variable_name="messages"),
    ] )

chain = ( RunnablePassthrough.assign(
    messages=itemgetter("messages") | trimmer)
) | prompt | model 


In [209]:
response = chain.invoke(
    {"messages": messages1 + [HumanMessage(content="What is my favourite ice cream")], "language": "English"})

response.content


'Vanilla! You mentioned it earlier.'

In [210]:
chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

In [211]:
response = chain.invoke(
    {"messages": messages + [HumanMessage(content="What problem did i ask for")], "language": "English"})

response.content


'You didn\'t exactly ask for a problem, but you asked if I was "having fun", and my response was an affirmative "yes!"'

In [213]:
## Wrapping this in a message history

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages")

config8 = {"configurable": {"session_id": "chat5"}}
response.content

'You didn\'t exactly ask for a problem, but you asked if I was "having fun", and my response was an affirmative "yes!"'